In [1]:
from dotenv import load_dotenv
import os
import ibis
from ibis import _
import pandas as pd
import numpy as np
import time
import re

In [2]:
from dash import Dash, dcc, html, Input, Output, callback,dash_table
from dash import State, no_update
from dash.exceptions import PreventUpdate
import json
import dash_bootstrap_components as dbc
from dash import ctx

In [3]:
# load env

load_dotenv(override=True)
kwargs_con = {
    'host':os.getenv('db_host_'),
    'user':os.getenv('db_user_'),
    'port':os.getenv('db_port_'),
    'password':os.getenv('db_password_'),
    'database':os.getenv('db_database_'),
    'driver':os.getenv('db_driver_')
}

kwargs_tbl_name = {
    'tbl_pr_':os.getenv('db_table_province_'),
    'tbl_car_':os.getenv('db_table_car_'),
    'tbl_sp_':os.getenv('db_table_sp_')
}

In [4]:
class IbisMSSQLConnection:
    """Context Manager for connect to MSSQL through ibis"""
    
    def __init__(self, connection_params):
        """
        Initialize connection manager
        
        Args:
            connection_params: connection parameters
        """
        self.connection_params = connection_params
        self.connection= None
    
    def __enter__(self):
        """ __enter__ part of context"""
        try:
            self.connection = ibis.mssql.connect(**self.connection_params)
            # print('--- Connected ---')
            return self.connection
        except Exception as e:
            raise ConnectionError(f"Error in connecting to database(__enter__): {e}")
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """close part of context"""
        if self.connection:
            try:
                self.connection.disconnect()
                # print('--- Disconnected ---')
            except Exception as close_error:
                print(f"⚠️ Error in closing the connection (__exit__) : {close_error}")
            finally:
                self.connection = None
        
        # اگر خطایی در context رخ داده بود، آن را suppress نکن
        # return False = “خطا را به بیرون منتقل کن (propagate)”
        # return True = “خطا را سرکوب کن (suppress)”
        #برای connection managerها معمولاً return False مناسب‌تر است چون می‌خواهیم خطاهای واقعی برنامه دیده شوند.
        if exc_type is not None:
            print(f"خطای {exc_type.__name__}: {exc_val}")
        
            # observing traceback
            import traceback
            traceback.print_tb(exc_tb)
        return False


In [8]:
try:
    conn = ibis.mssql.connect(**kwargs_con)
    print('--- Connected ---')
except Exception as e:
    raise ConnectionError(f"Error in connecting to database(__enter__): {e}")

--- Connected ---


In [24]:
def load_list_tables():
    try:
        with IbisMSSQLConnection(kwargs_con) as con:
            return list(con.list_tables())
    except Exception as e:
        print(e)
        return []

def load_columns(tbl):
    with IbisMSSQLConnection(kwargs_con) as con:
        return list(con.table(tbl).columns)
# load_columns("FactContract")

def load_data(tbl_,clmns_,limit_):
    limit = int(limit_)
    with IbisMSSQLConnection(kwargs_con) as con:
        tbl_con = con.table(tbl_).order_by(ibis.desc("MiladiInternalDate"))
        data = tbl_con.select(clmns_).limit(limit_).execute()
    return data

In [47]:
conn.disconnect()

In [6]:
tbl_ = "FactContract"
clmns_ = ["ContractStatusId","ContractID"]
load_data(tbl_,clmns_,3)

,ContractStatusId,ContractID
0,None,31305575
1,None,39372511
2,None,39432690


In [15]:
tbl_con = conn.table(tbl_)
tbl_con.select(clmns_).limit(5).execute()

,ContractStatusId,ContractID
0,None,31305575
1,None,39372511
2,None,39432690
3,None,39713090
4,None,40059595


In [29]:
kwargs_defaults_datatbl = {
    "column_selectable":"multi",
    "editable":True,
    "export_format": "csv",
    "export_headers": "display",    
    "filter_action":"native",    
    "page_action": "native",
    "page_size": 5,
    "row_selectable":"multi",
    "style_table": {"overflowX": "auto"},
    "sort_action":"native"
    }
# kwargs_defaults_datatbl= {}



app = Dash(__name__ ,
    external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container(
    [
        dbc.Row(
            [
                dbc.Col(
                    dcc.Dropdown(
                        id="Id_load_list_tables",
                        multi=False,
                    ),
                    width=3,
                ),
                dbc.Col(
                    dcc.Dropdown(
                        id="Id_filter_columns",
                        multi=True,
                    ),
                    width=3,
                ),
                
                dbc.Col(
                    dcc.Input(
                        id="Id_limit",
                        type="number",
                        min=1,
                        max=1000,
                        step=1,
                        value=5,
                    ),
                    width=2,
                ),
                
            ],
            className="mb-3",
        ),

        dbc.Row(
            [
                
                dbc.Col(
                    dcc.Dropdown(
                        id="Id_load_list_tables_2",
                        multi=False,
                    ),
                    width=3,
                ),
                dbc.Col(
                    dcc.Dropdown(
                        id="Id_filter_columns_2",
                        multi=True,
                    ),
                    width=3,
                ),
                dbc.Col(
                    dcc.Input(
                        id="Id_limit_2",
                        type="number",
                        min=1,
                        max=1000,
                        step=1,
                        value=5,
                    ),
                    width=2,
                ),
            ],
            className="mb-3",
        ),


        

        dbc.Row(
            [
                dbc.Col(
                    dash_table.DataTable(
                        id="Id_tbl", **kwargs_defaults_datatbl
                        
                    ),
                    width=12,
                )
            ]
        ),

        dbc.Row(
            [               
                dbc.Col(
                    dash_table.DataTable(
                        id="Id_tbl_2", **kwargs_defaults_datatbl
                    ),
                    width=12,
                ),
            ]
        ),

        
    ],
    fluid=True,
)


@callback(
    Output('Id_load_list_tables', 'options'),
    Output('Id_load_list_tables', 'value'),
    Input('Id_load_list_tables', 'id')
)
def init_load_list_tables(id_):
    list_tbl = load_list_tables()
    options = [{'label': r, 'value': r} for r in list_tbl]
    # value = options[0]['value'] if options else None
    value = "FactContract"
    return options, value


@callback(
    Output('Id_filter_columns', 'options'),
    Output('Id_filter_columns', 'value'),
    Input('Id_load_list_tables', 'value')    
)
def init_load_columns(tbl):
    if not tbl:
        return [], []
    clmns = load_columns(tbl)
    options = [{'label': r, 'value':r} for r in clmns]
    value = [r for r in clmns]
    return options, value

@callback(
    Output('Id_filter_columns_2', 'options'),
    Output('Id_filter_columns_2', 'value'),
    Input('Id_load_list_tables_2', 'value')    
)
def init_load_columns_2(tbl):
    if not tbl:
        return [], []    
    clmns = load_columns(tbl)
    options = [{'label': r, 'value':r} for r in clmns]
    value = [r for r in clmns]
    return options, value
    

@callback(
    Output('Id_load_list_tables_2', 'options'),
    Output('Id_load_list_tables_2', 'value'),
    Input('Id_load_list_tables_2', 'id')
)
def init_load_list_tables_2(id_):
    list_tbl = load_list_tables()
    options = [{'label': r, 'value': r} for r in list_tbl]
    # value = options[0]['value'] if options else None
    value = "FactContract"
    return options, value


@app.callback(
    Output("Id_tbl", "columns"),
    Output("Id_tbl", "data"),
    Input("Id_load_list_tables","value"),
    Input('Id_filter_columns', 'value'),
    Input("Id_limit","value")
)
def init_data_tbl(tbl_,clmns_,limit_):
    if not tbl_:
        return [], []
    if not clmns_:
        return [], []
    # if isinstance(clmns_, str):
        # clmns_ = [clmns_]     
    df = load_data(tbl_,clmns_,limit_)
    columns = [{"name": c, "id": c} for c in df.columns]
    return columns, df.to_dict("records")

@app.callback(
    Output("Id_tbl_2", "columns"),
    Output("Id_tbl_2", "data"),
    Input("Id_load_list_tables_2","value"),
    Input('Id_filter_columns_2', 'value'),
    Input("Id_limit_2","value")
)
def init_data_tbl(tbl_,clmns_,limit_):
    if not tbl_:
        return [], []
    if not clmns_:
        return [], []
    # if isinstance(clmns_, str):
        # clmns_ = [clmns_]     
    df = load_data(tbl_,clmns_,limit_)
    columns = [{"name": c, "id": c} for c in df.columns]
    return columns, df.to_dict("records")

   
if __name__ == '__main__':
    app.run(debug=True,port=9873)